# Module 2.6 — Domain-Specific Evaluation
**DeskMate SLM Curriculum · Phase 2 · Notebook 13**

Run on **CPU** — inference on a small gold set needs no GPU.

By the end you will have:
- Per-class precision / recall / F1 on the gold set
- A calibration (reliability) diagram and ECE score
- PR curves with threshold recommendations for critical intents
- A ranked top-10 error pattern analysis
- `reports/eval_report.md` summarising all findings

Read `2.6_evaluation.md` for full theory.

---

## Step 0 — Install & Seed

In [ ]:
%%capture
!pip install -q transformers==4.44.0 datasets==2.21.0 torch==2.3.1 scikit-learn==1.5.1 matplotlib==3.9.0

In [ ]:
import random, json, pathlib, textwrap
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, f1_score, accuracy_score,
)
from sklearn.calibration import calibration_curve
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Libraries loaded.')

## Step 1 — Runtime & Paths

In [ ]:
try:
    import google.colab; RUNTIME = 'colab'
    PROJECT_ROOT = pathlib.Path('/content/slm')
except ImportError:
    try:
        import kaggle_secrets; RUNTIME = 'kaggle'
        PROJECT_ROOT = pathlib.Path('/kaggle/working/slm')
    except ImportError:
        RUNTIME = 'local'
        PROJECT_ROOT = pathlib.Path('.').resolve()

DATA_PROC  = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS    = PROJECT_ROOT / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)
print(f'Runtime : {RUNTIME}')
print(f'Reports : {REPORTS}')

## Step 2 — Label Maps

In [ ]:
maps_path = DATA_PROC / 'label_maps.json'
if maps_path.exists():
    lm = json.loads(maps_path.read_text())
    INTENT2ID   = lm['INTENT2ID']
    ID2INTENT   = {int(k): v for k, v in lm['ID2INTENT'].items()}
    PRIORITY2ID = lm['PRIORITY2ID']
    ID2PRIORITY = {int(k): v for k, v in lm['ID2PRIORITY'].items()}
else:
    INTENTS     = ['account_access','account_settings','billing_dispute','billing_inquiry',
                   'cancellation','data_privacy','feature_request','onboarding',
                   'outage_report','payment_failure','performance_issue','refund_request',
                   'technical_bug','usage_question','out_of_scope']
    INTENT2ID   = {i: n for n, i in enumerate(INTENTS)}
    ID2INTENT   = {n: i for n, i in enumerate(INTENTS)}
    PRIORITIES  = ['low','medium','high']
    PRIORITY2ID = {p: n for n, p in enumerate(PRIORITIES)}
    ID2PRIORITY = {n: p for n, p in enumerate(PRIORITIES)}

N_INTENTS = len(INTENT2ID)
print(f'Intent classes: {N_INTENTS}')

## Step 3 — Load Gold Set

The gold set is frozen and was never used for training or augmentation.
A stratified placeholder is generated if the file is missing.

In [ ]:
def make_gold_placeholder(n_per_class=5):
    intents = list(INTENT2ID.keys())
    prios   = list(PRIORITY2ID.keys())
    rows    = []
    templates = [
        'I have a problem with {intent} on my account.',
        'Please help me with {intent}.',
        'I need support for {intent} issue.',
        'Unable to complete {intent} process.',
        'Urgent: {intent} not working.',
    ]
    for intent in intents:
        for i in range(n_per_class):
            rows.append({
                'text':     templates[i].format(intent=intent.replace('_', ' ')),
                'intent':   intent,
                'priority': prios[i % len(prios)],
                'source':   'placeholder',
            })
    return rows

gold_path = DATA_PROC / 'gold.jsonl'
if not gold_path.exists():
    print('WARNING: gold.jsonl not found — using stratified placeholder (run Module 2.1 first)')
    rows = make_gold_placeholder(n_per_class=5)
    DATA_PROC.mkdir(parents=True, exist_ok=True)
    gold_path.write_text('\n'.join(json.dumps(r) for r in rows))

gold_df = pd.read_json(gold_path, lines=True)
print(f'Gold set: {len(gold_df)} examples')
print(f'Columns : {list(gold_df.columns)}')
print(f'Intent distribution (top 5):')
print(gold_df['intent'].value_counts().head())

## Step 4 — Run Intent Classifier on Gold Set

We load the fine-tuned model from Module 2.4. If it is not present,
a random-weight model is used so all downstream analysis cells still run.

In [ ]:
MODEL_NAME   = 'microsoft/MiniLM-L12-H384-uncased'
tokenizer    = AutoTokenizer.from_pretrained(MODEL_NAME)
model_path   = MODELS_DIR / 'intent_classifier'

if model_path.exists():
    intent_model = AutoModelForSequenceClassification.from_pretrained(str(model_path))
    print(f'Loaded fine-tuned intent model from {model_path}')
else:
    print('WARNING: intent_classifier not found — using random weights (train Module 2.4 first)')
    intent_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=N_INTENTS, id2label=ID2INTENT, label2id=INTENT2ID)

intent_model.eval()

def batch_predict(texts, model, tokenizer, batch_size=32):
    all_logits = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        enc = tokenizer(batch_texts, truncation=True, padding=True,
                        max_length=128, return_tensors='pt')
        with torch.no_grad():
            logits = model(**enc).logits
        all_logits.append(logits)
    return torch.cat(all_logits, dim=0)

texts         = gold_df['text'].tolist()
logits        = batch_predict(texts, intent_model, tokenizer)
probas        = logits.softmax(dim=-1).numpy()
y_pred        = logits.argmax(dim=-1).numpy()
y_true        = np.array([INTENT2ID.get(i, 0) for i in gold_df['intent']])

print(f'Accuracy : {accuracy_score(y_true, y_pred)*100:.1f}%')
print(f'Macro-F1 : {f1_score(y_true, y_pred, average="macro", zero_division=0)*100:.1f}%')

## Step 5 — Per-Class F1 Table

In [ ]:
class_names = [ID2INTENT[i] for i in range(N_INTENTS)]
report = classification_report(
    y_true, y_pred, target_names=class_names, zero_division=0, output_dict=True)

rows = []
for cls in class_names:
    m = report[cls]
    rows.append({'intent': cls,
                 'precision': round(m['precision'], 3),
                 'recall':    round(m['recall'], 3),
                 'f1':        round(m['f1-score'], 3),
                 'support':   int(m['support'])})

report_df = pd.DataFrame(rows).sort_values('f1')
print('Per-class F1 (sorted by F1 ascending — worst classes first):')
print(report_df.to_string(index=False))

## Step 6 — Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(13, 10))
cm      = confusion_matrix(y_true, y_pred)
short   = [n[:12] for n in class_names]
ConfusionMatrixDisplay(cm, display_labels=short).plot(
    ax=ax, xticks_rotation=45, colorbar=False, cmap='Blues')
ax.set_title('Intent Classifier — Confusion Matrix (gold set)')
plt.tight_layout()
cm_path = REPORTS / 'intent_confusion_gold.png'
plt.savefig(str(cm_path), bbox_inches='tight')
plt.show()
print(f'Saved: {cm_path}')

In [ ]:
# Top confusion pairs (off-diagonal with highest counts)
pairs = []
for i in range(N_INTENTS):
    for j in range(N_INTENTS):
        if i != j and cm[i, j] > 0:
            pairs.append((cm[i, j], class_names[i], class_names[j]))
pairs.sort(reverse=True)
print('Top-10 confusion pairs (true → predicted):')
for cnt, true_lbl, pred_lbl in pairs[:10]:
    print(f'  {true_lbl:<25} → {pred_lbl:<25} x{cnt}')

## Step 7 — Calibration (Reliability Diagram)

A well-calibrated model has predicted probability ≈ empirical accuracy.
Points below the diagonal = overconfident.

In [ ]:
# One-vs-rest calibration: max predicted probability vs correctness
max_prob   = probas.max(axis=1)
is_correct = (y_pred == y_true).astype(int)

frac_pos, mean_conf = calibration_curve(is_correct, max_prob, n_bins=10, strategy='quantile')

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0,1],[0,1], 'k--', label='Perfect calibration')
ax.plot(mean_conf, frac_pos, 'o-', color='#4C72B0', label='Intent classifier')
ax.set_xlabel('Mean predicted confidence')
ax.set_ylabel('Fraction correct')
ax.set_title('Reliability Diagram (gold set)')
ax.legend()
plt.tight_layout()
plt.savefig(str(REPORTS / 'calibration.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Expected Calibration Error
n      = len(y_true)
ece    = 0.0
n_bins = 10
bins   = np.linspace(0, 1, n_bins + 1)
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (max_prob >= lo) & (max_prob < hi)
    if mask.sum() == 0:
        continue
    acc  = is_correct[mask].mean()
    conf = max_prob[mask].mean()
    ece += (mask.sum() / n) * abs(acc - conf)

print(f'ECE = {ece:.4f}  (0 = perfect, >0.05 warrants recalibration)')
if ece > 0.05:
    print('  → Consider temperature scaling or Platt scaling before production deployment.')
else:
    print('  → Calibration acceptable for routing use.')

## Step 8 — PR Curves & Threshold Tuning

Find the minimum confidence threshold that achieves recall ≥ 0.95 on critical intents.

In [ ]:
CRITICAL_INTENTS = ['outage_report', 'billing_dispute']
TARGET_RECALL    = 0.95

fig, axes = plt.subplots(1, len(CRITICAL_INTENTS), figsize=(12, 4))

thresholds_out = {}
for ax, intent_name in zip(axes, CRITICAL_INTENTS):
    cls_id        = INTENT2ID[intent_name]
    y_bin         = (y_true == cls_id).astype(int)
    prec, rec, thresh = precision_recall_curve(y_bin, probas[:, cls_id])

    ax.plot(rec, prec, color='#4C72B0')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'PR Curve: {intent_name}')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

    # Find threshold for target recall
    idx = None
    for i, r in enumerate(rec[:-1]):
        if r >= TARGET_RECALL:
            idx = i; break
    if idx is not None:
        t = thresh[idx]
        thresholds_out[intent_name] = round(float(t), 3)
        ax.axvline(TARGET_RECALL, color='red', linestyle='--',
                   label=f'recall={TARGET_RECALL}')
        ax.axhline(prec[idx], color='orange', linestyle='--',
                   label=f'prec={prec[idx]:.2f} @ t={t:.2f}')
        ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(str(REPORTS / 'pr_curves.png'), bbox_inches='tight')
plt.show()

print('Threshold recommendations for recall >= 0.95:')
for intent_name, t in thresholds_out.items():
    print(f'  {intent_name:<25}: confidence threshold = {t}')

## Step 9 — Error Analysis

Categorise misclassified examples into failure modes.
Read up to 20 errors and assign a category manually (or use heuristics).

In [ ]:
errors_df = gold_df.copy()
errors_df['y_true']    = [ID2INTENT[i] for i in y_true]
errors_df['y_pred']    = [ID2INTENT[i] for i in y_pred]
errors_df['max_conf']  = probas.max(axis=1).round(3)
errors_df = errors_df[errors_df['y_true'] != errors_df['y_pred']].copy()

print(f'Total errors: {len(errors_df)} / {len(gold_df)} ({len(errors_df)/len(gold_df)*100:.1f}%)')
print()
print('Sample errors (first 10):')
for _, row in errors_df.head(10).iterrows():
    print(f'  TRUE: {row["y_true"]:<22}  PRED: {row["y_pred"]:<22}  CONF: {row["max_conf"]}')
    print(f'  TEXT: {row["text"][:80]}')
    print()

In [ ]:
# Heuristic failure category assignment
# In a real analysis you would read each error and label manually.
# These heuristics approximate the most common patterns.

CATEGORIES = [
    'phrasing_gap',
    'ambiguous_ticket',
    'synthetic_artefact',
    'rare_class_underfit',
    'label_noise',
]

def assign_category(row):
    true_lbl, pred_lbl = row['y_true'], row['y_pred']
    conf = row['max_conf']
    # High-confidence wrong prediction → likely phrasing gap or artefact
    if conf > 0.85:
        return 'synthetic_artefact'
    # Low-confidence → ambiguous or rare class
    if conf < 0.5:
        rare = {'data_privacy','outage_report','cancellation'}
        if true_lbl in rare:
            return 'rare_class_underfit'
        return 'ambiguous_ticket'
    # Medium confidence — typical phrasing gap
    return 'phrasing_gap'

errors_df['category'] = errors_df.apply(assign_category, axis=1)

cat_counts = errors_df['category'].value_counts()
print('Failure category distribution:')
for cat, cnt in cat_counts.items():
    bar = '#' * cnt
    print(f'  {cat:<25}: {cnt:>3}  {bar}')

In [ ]:
# Top-10 error patterns: (true, pred) pair + category + count
pattern_counts = (
    errors_df.groupby(['y_true','y_pred','category'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
print('Top-10 error patterns:')
print(pattern_counts.to_string(index=False))

## Step 10 — Write Evaluation Report

In [ ]:
macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
accuracy = accuracy_score(y_true, y_pred)

lines = [
    '# DeskMate Evaluation Report\n',
    '## Summary\n',
    f'- Gold set size : {len(gold_df)}',
    f'- Accuracy      : {accuracy*100:.1f}%',
    f'- Macro-F1      : {macro_f1*100:.1f}%',
    f'- ECE           : {ece:.4f}',
    f'- Total errors  : {len(errors_df)}\n',
    '## Per-Class F1 (bottom 5 — worst)\n',
]
for _, row in report_df.head(5).iterrows():
    lines.append(f'- {row["intent"]:<25}: F1={row["f1"]:.3f}  P={row["precision"]:.3f}  R={row["recall"]:.3f}')

lines += [
    '',
    '## Calibration\n',
    f'- ECE = {ece:.4f}',
    '- See `reports/calibration.png` for reliability diagram\n',
    '## Threshold Recommendations\n',
]
for intent_name, t in thresholds_out.items():
    lines.append(f'- {intent_name}: threshold = {t} for recall >= 0.95')

lines += ['', '## Top-10 Error Patterns\n']
for _, row in pattern_counts.iterrows():
    lines.append(f'- [{row["count"]}x] {row["y_true"]} → {row["y_pred"]} ({row["category"]})')

lines += [
    '',
    '## Failure Category Summary\n',
]
for cat, cnt in cat_counts.items():
    lines.append(f'- {cat}: {cnt} errors')

report_path = REPORTS / 'eval_report.md'
report_path.write_text('\n'.join(lines))
print(f'Eval report written: {report_path}')
print()
print('\n'.join(lines[:30]))

## Checkpoint

> **When is macro-F1 the right metric over accuracy?**

Write your answer below. A strong answer covers:
1. Why imbalanced classes make accuracy misleading.
2. What macro-F1 weights equally that accuracy does not.
3. A concrete example from DeskMate.

In [ ]:
answer = """
[Write your answer here]
"""
print(answer)

## Deliverable Summary

| Artifact | Location |
|---|---|
| Evaluation report | `reports/eval_report.md` |
| Confusion matrix | `reports/intent_confusion_gold.png` |
| Calibration diagram | `reports/calibration.png` |
| PR curves | `reports/pr_curves.png` |

**Phase 2 complete.** You have:
- A labelled gold dataset (2.1)
- Augmented synthetic training data (2.2)
- A tokenized, collated DatasetDict (2.3)
- An intent + priority classifier (2.4)
- A field extractor (NER) (2.5)
- A rigorous evaluation report on the gold set (2.6)

**Next options:**
- Module 2.7 (optional) — Distillation: shrink the classifier with knowledge distillation.
- Phase 3 — The decoder SLM: domain-adapted generation, PEFT/LoRA, and structured output.